# Scaled Dot-Product Attention

Wiki reference for [scaled dot-product attention](https://ml-viz-ruby.vercel.app/wiki/scaled-dot-product-attention).

**The idea in one sentence.** Attention scores each query against every key by a dot product,
**scaled by $1/\sqrt{d_k}$**, then softmaxes into weights that mix the values — and the scaling
is essential because the dot product's variance grows linearly with $d_k$, so without it softmax
would saturate.

We implement scaled dot-product attention from scratch, **validate the variance growth and the
weight distribution**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')

## From-scratch attention

In [ ]:
def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))  # numerically stable
    return e / e.sum(axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)     # (n, n)
    if mask is not None:
        scores[mask] = -1e9
    weights = softmax(scores)            # row-wise softmax
    return weights @ V, weights

# Wiki worked example: 3 tokens, d_k=2
Q = np.array([[2,0],[0,2],[2,2]], dtype=float)
K = np.array([[2,0],[0,2],[1,1]], dtype=float)
V = np.array([[1,0],[0,1],[10,10]], dtype=float)

out, W = scaled_dot_product_attention(Q, K, V)
print("Attention weights (3×3):")
print(W.round(3))
print("\nOutput (3×2):")
print(out.round(3))
# Verify: row 1 ≈ [2.635, 1.912]; row 3 ≈ [3.667, 3.667]

## Variance grows with d_k — the √d_k fix

In [ ]:
d_values = [2, 8, 32, 64, 128]
n_samples = 20_000
variances = []
for d in d_values:
    q = np.random.randn(n_samples, d)
    k = np.random.randn(n_samples, d)
    dots = (q * k).sum(axis=1)
    variances.append(dots.var())

plt.figure(figsize=(7,4))
plt.plot(d_values, variances, 'o-', color='#6366f1', label='Empirical Var(q·k)')
plt.plot(d_values, d_values, '--', color='#f59e0b', label='Expected = d_k')
plt.xlabel('d_k'); plt.ylabel('Variance'); plt.title('Why we scale by 1/√d_k')
plt.legend(); plt.tight_layout(); plt.show()

### Validate: the dot product's variance grows with $d_k$

For unit-variance random $q, k$, the dot product $q\cdot k$ has variance $\approx d_k$. That's why
the raw scores get large in high dimensions — and exactly why we divide by $\sqrt{d_k}$. We
confirm the empirical variance tracks $d_k$.

In [ ]:
print('Var(q.k):', [round(v, 1) for v in variances], 'vs d:', list(d_values))
for v, d in zip(variances, d_values):
    assert abs(v - d) / d < 0.1, 'Var(q.k) grows linearly with d_k'
print('\n✅ dot-product variance = d_k -> scale by 1/sqrt(d_k) to keep scores in range')

## Attention heatmap on a 4-token sequence

In [ ]:
rng = np.random.default_rng(42)
n, d_k = 6, 8
Q_r = rng.standard_normal((n, d_k))
K_r = rng.standard_normal((n, d_k))
V_r = rng.standard_normal((n, d_k))

_, W_r = scaled_dot_product_attention(Q_r, K_r, V_r)

tokens = ['The','cat','sat','on','the','mat']
plt.figure(figsize=(6,5))
plt.imshow(W_r, vmin=0, vmax=1, cmap='Blues', aspect='auto')
plt.colorbar(label='Attention weight')
plt.xticks(range(n), tokens, rotation=45)
plt.yticks(range(n), tokens)
plt.title('Attention weights (random projections)')
plt.tight_layout(); plt.show()

### Validate: attention weights form a distribution

The row-wise softmax makes each query's attention weights **non-negative and sum to 1**, so the
output is a convex combination of the value vectors. We confirm each row is a valid
distribution.

In [ ]:
print('row sums:', W_r.sum(axis=1).round(4))
assert np.allclose(W_r.sum(axis=1), 1.0), 'each query attends with a probability distribution (softmax rows sum to 1)'
assert (W_r >= 0).all(), 'attention weights are non-negative'
print('\n✅ attention output is a weighted average of values, weights from softmax')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **missing $1/\sqrt{d_k}$** | softmax saturates, gradients vanish (demo) |
| **no numerical stabilization** | subtract the max before exp to avoid overflow |
| **quadratic cost** | attention is $O(n^2)$ in sequence length |
| **masking** | causal/padding masks must set scores to $-\infty$ before softmax |
| **fp16 softmax** | needs care; use stable / flash kernels |

Demo: the $1/\sqrt{d_k}$ scaling keeps the attention distribution from collapsing.

In [ ]:
# Why the 1/sqrt(d_k) scaling is not optional: with large d_k the raw dot products have large
# variance, so softmax saturates onto a single key -> the attention distribution collapses (low
# entropy) and its gradient nearly vanishes. Dividing by sqrt(d_k) keeps scores in a sane range.
# We compare the attention entropy with and without the scaling at d_k=128.
d_k = 128
Qb = np.random.randn(4, d_k)
Kb = np.random.randn(4, d_k)
raw = Qb @ Kb.T
scaled = raw / np.sqrt(d_k)
ent = lambda s: float(-(softmax(s) * np.log(softmax(s) + 1e-12)).sum(-1).mean())
print(f'attention entropy: unscaled = {ent(raw):.3f},  scaled = {ent(scaled):.3f}')
assert ent(scaled) > ent(raw), 'without 1/sqrt(d_k), large dot products saturate softmax (low entropy)'
print('\nLarge d_k inflates score variance -> softmax saturates. The 1/sqrt(d_k) factor prevents the collapse.')

## ✏️ Your turn

**Task:** Implement **causal masking** — add $-\infty$ to all upper-triangle entries of the score matrix before softmax, so token $t$ can only attend to positions $\le t$.

Verify that after masking, the upper triangle of the weight matrix is exactly 0.

In [ ]:
# TODO(you): create a causal mask (upper triangle, diagonal=1)
# mask = np.triu(np.ones((n, n), dtype=bool), k=1)
# out_causal, W_causal = scaled_dot_product_attention(Q_r, K_r, V_r, mask=mask)

In [ ]:
# assert np.allclose(W_causal[np.triu(np.ones((n,n),dtype=bool), k=1)], 0, atol=1e-6)

<details><summary>Solution</summary>

```python
mask = np.triu(np.ones((n,n), dtype=bool), k=1)
out_c, W_c = scaled_dot_product_attention(Q_r, K_r, V_r, mask=mask)
print('Upper-triangle weights (should be ~0):')
print(W_c[mask].round(6))
assert np.allclose(W_c[mask], 0, atol=1e-6)
print('Causal mask verified ✓')
```
</details>

## Key takeaways

- **Attention = softmax(QK^T / sqrt(d_k)) V** — scores, normalize, mix values.
- **Var(q·k) = d_k** (verified) — the reason for the scaling.
- **Weights are a distribution** per query (verified) — the output is a convex mix of values.
- **The $1/\sqrt{d_k}$ scaling** keeps softmax out of saturation (demo).